In [12]:
import numpy as np
import statsmodels.api as sm
from ISLP import load_data
from ISLP.models import (ModelSpec as MS,
summarize,
poly)
from sklearn.model_selection import train_test_split

from functools import partial
from sklearn.model_selection import \
(cross_validate,
KFold,
ShuffleSplit)
from sklearn.base import clone
from ISLP.models import sklearn_sm

### The Validation Set Approach

In [13]:
auto = load_data("Auto")
len(auto)

392

In [14]:
half = len(auto) // 2

auto_train, auto_valid = train_test_split(auto, 
                                          test_size=half,
                                          random_state=0)

In [15]:
design = MS(["horsepower"])
X_train = design.fit_transform(auto_train)
y_train = auto_train["mpg"]

model = sm.OLS(y_train, X_train)
results = model.fit()

In [16]:
X_valid = design.fit_transform(auto_valid)
y_valid = auto_valid["mpg"]

valid_pred = results.predict(X_valid)

np.mean((y_valid - valid_pred)**2)

23.61661706966988

In [ ]:
def evalMSE(terms, train, test, response):
  mm = MS(terms)
  X_train = mm.fit_transform(train)
  y_train = train[response]
  X_test = mm.transform(test)
  y_test = test[response]
  
  results = sm.OLS(y_train, X_train).fit()
  test_pred = results.predict(X_test)
  
  return np.mean((y_test - test_pred)**2)

In [23]:
MSE = np.zeros(3)
for idx, degree in enumerate(range(1, 4)):
 MSE[idx] = evalMSE([poly('horsepower', degree)], 
                    auto_train, 
                    auto_valid, 
                    'mpg')
 
MSE

array([23.61661707, 18.76303135, 18.79694163])

In [24]:
auto_train, auto_test = train_test_split(auto, test_size=half, random_state=3)

MSE = np.zeros(3)
for idx, degree in enumerate(range(1, 4)):
 MSE[idx] = evalMSE([poly('horsepower', degree)], 
                    auto_train, 
                    auto_valid, 
                    'mpg')
 
MSE

array([23.55461807, 18.82812083, 18.8662578 ])

Using this split of the observations into a training set and a validation
set, we find that the validation set error rates for the models with linear,
quadratic, and cubic terms are 20.76, 16.95, and 16.97, respectively.

These results are consistent with our previous findings: a model that
predicts mpg using a quadratic function of horsepower performs better than
a model that involves only a linear function of horsepower, and there is no
evidence of an improvement in using a cubic function of horsepower.

### Cross-Validation

In [30]:
hp_model = sklearn_sm(sm.OLS, MS(['horsepower']))
X, Y = auto.drop(columns=['mpg']), auto['mpg']

cv_results = cross_validate(hp_model, X, Y, cv=auto.shape[0])
cv_error = np.mean(cv_results["test_score"])
cv_error

24.23151351792922

Above we see an example of LOOCV

In [31]:
cv_error = np.zeros(5)
H = np.array(auto['horsepower'])
M = sklearn_sm(sm.OLS)

for i, d in enumerate(range(1,6)):
  X = np.power.outer(H, np.arange(d+1))  #create matrix of polynomials coeffs: h + h^2 + ... h^5
  M_CV = cross_validate(M,
                        X,
                        Y,
                        cv=auto.shape[0])
  cv_error[i] = np.mean(M_CV['test_score'])
cv_error

array([24.23151352, 19.24821312, 19.33498406, 19.4244303 , 19.03322411])

In [34]:
cv_error = np.zeros(5)
cv = KFold(n_splits=10, shuffle=True, random_state=0)

for i, d in enumerate(range(1, 6)):
  X = np.power.outer(H, np.arange(d + 1))
  M_CV = cross_validate(M,
                        X,
                        Y,
                        cv=cv)
  cv_error[i] = np.mean(M_CV["test_score"])
cv_error


array([24.20766449, 19.18533142, 19.27626666, 19.47848402, 19.13719154])

In [35]:
validation = ShuffleSplit(n_splits=1,
                          test_size=196,
                          random_state=0)

results = cross_validate(hp_model,
                         auto.drop(columns="mpg"),
                         auto["mpg"],
                         cv=validation)
results["test_score"]

array([23.61661707])

In [36]:
validation = ShuffleSplit(n_splits=10,
                          test_size=196,
                          random_state=0)

results = cross_validate(hp_model,
                         auto.drop(columns="mpg"),
                         auto["mpg"],
                         cv=validation)
np.mean(results["test_score"]), results["test_score"].std()

(23.802232661034164, 1.4218450941091831)

### The Bootstrap

In [37]:
portfolio = load_data("Portfolio")
portfolio

,X,Y
0,-0.895251,-0.234924
1,-1.562454,-0.885176
2,-0.417090,0.271888
3,1.044356,-0.734198
4,-0.315568,0.841983
...,...,...
95,0.479091,1.454774
96,-0.535020,-0.399175
97,-0.773129,-0.957175
98,0.403634,1.396038


$$\alpha=\frac{\sigma_Y^2-\sigma_{XY}}{\sigma_X^2+\sigma_Y^2-2\sigma_{XY}}$$

is used when we want to find the value of α that minimizes the variance of a linear combination of two random variables X and Y, typically something like:
$$Z=\alpha X+(1-\alpha)Y$$

In [43]:
import pandas as pd

D = pd.DataFrame({'X': [1, 2, 3], 'Y': [2, 3, 6]})
cov_ = np.cov(D[['X','Y']], rowvar=False)

cov_


array([[1.        , 2.        ],
       [2.        , 4.33333333]])

Above we see covariance matrix:
- [0, 0] - variance of x
- [1, 1] - variance of y
- [0, 1] - covariance of (x, y)
- [1, 0] - covariance of (x, y)

In [48]:
def alpha_func(D, idx):
    cov_ = np.cov(D[["X", "Y"]].loc[idx], rowvar=False)
    return ((cov_[1, 1] - cov_[0, 1]) /
            (cov_[0, 0] + cov_[1, 1] - 2 * cov_[0, 1]))

In [46]:
portfolio.shape

(100, 2)

In [50]:
alpha_func(portfolio, range(portfolio.shape[0]))

0.57583207459283

In [51]:
rng = np.random.default_rng(0)
alpha_func(portfolio, rng.choice(100, 100, replace=True))

0.6074452469619004

In [53]:
def boot_SE(func, D, n=None, B=1000, seed=0):
    rng = np.random.default_rng(seed)
    first_, second_ = 0, 0
    n = n or D.shape[0]
    for _ in range(B):
        idx = rng.choice(D.index, n, replace=True)
        value = func(D, idx)
        first_ += value
        second_ += value**2
    return np.sqrt(second_ / B - (first_ / B)**2)


$$\mathrm{SE}_{\mathrm{bootstrap}}=\sqrt{\frac{1}{B}\sum_{i=1}^B\theta_i^2-\left(\frac{1}{B}\sum_{i=1}^B\theta_i\right)^2}$$

In [54]:
alpha_SE = boot_SE(alpha_func, portfolio, B=1000, seed=0)
alpha_SE

0.09118176521277699